# Interactive Parameter Exploration — Sample-1

This notebook lets you interactively tune the main parameters of DelaDect's
crack and delamination (edge + diffuse) detectors and see the effect
immediately on real data from `example_images/sample-1`.

Sample-1 is a `[0, 90]` cross-ply laminate with 5 sparse frames:
`0002_sc.png`, `0092_sc.png`, `0189_sc.png`, `0284_sc.png`, `0378_sc.png`.
Frames `0002`/`0092` are early in the test and show little to no visible
damage; damage becomes visible from `0189_sc.png` onward, so the sliders
below default to `0378_sc.png` (the most damaged frame) so you see
something immediately.

**Sections:**
1. Crack detection — `crack_width_px`, `min_crack_size_px`
2. Edge delamination — `hard_floor`, `window_edge`, `seed_ratio`, `post_threshold_closing_scale`
3. Diffuse delamination — `diffuse_dx`/`diffuse_dy`, `hard_floor`, `post_threshold_closing_scale`

**Speed note:** crack detection (Section 1) takes roughly 15-25 seconds per
run (skimage/numba cost, not something this notebook can hide), so it uses
a **Run** button rather than a live slider. Edge and diffuse delamination
(Sections 2-3) run on a *single selected frame* at a time and update live
as you drag the sliders.

**Environment:** this notebook needs DelaDect's real dependency set
(NumPy < 2 — see the main `README.md`) plus `ipywidgets` and `ipympl`. If
sliders don't render, run `pip install ipywidgets ipympl` and restart the
kernel.

In [ ]:
%matplotlib widget
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import interact, interact_manual, widgets

from deladect.specimen import Specimen
from deladect.detection import DelaminationDetector, crack_eval_crossply

warnings.filterwarnings("ignore")  # crackdect/numba deprecation chatter, not relevant here

# Resolve the repo root whether this notebook is run from its own folder
# (notebooks/) or from the repo root.
if (Path.cwd() / "example_images").exists():
    ROOT = Path.cwd()
elif (Path.cwd().parent / "example_images").exists():
    ROOT = Path.cwd().parent
else:
    raise FileNotFoundError(
        "Could not find example_images/ -- run this notebook from the repo root or from notebooks/."
    )

DATA_ROOT = ROOT / "example_images" / "sample-1"
FRAME_NAMES = sorted(p.name for p in DATA_ROOT.glob("*.png"))
FRAME_INDEX = {name: i for i, name in enumerate(FRAME_NAMES)}
print("Frames:", FRAME_NAMES)

## 1) Load sample-1 into a `Specimen`

`Specimen.from_cross_ply` builds a `[0, 90]` cross-ply specimen and
automatically adds the `"0/90"` interface between the two plies.

In [ ]:
specimen = Specimen.from_cross_ply(
    name="sample-1-interactive",
    scale_px_mm=41.03328366,
    path_full=str(DATA_ROOT),
    sorting_key="_sc",
    image_types=["png"],
    results_root=str(ROOT / "results"),
    avg_crack_width_px=8.0,
)
interface = specimen.interfaces[0]
print(f"Loaded {len(specimen.path_full_list)} frames; interface: {interface.name!r}")

## 2) Baseline crack run

Diffuse delamination detection uses crack segments to define its
region-of-interest, so we need a crack pass before diffuse detection can
run at all. This uses the library's own defaults once; Section 1 below
lets you re-run it with your own `crack_width_px`/`min_crack_size_px`.

This takes roughly 15-25 seconds.

In [ ]:
print("Running baseline crack_eval_crossply()... (~15-25s)")
crack_results = crack_eval_crossply(specimen, export_images=False, save_cracks=False)
cracks = Specimen.join_cracks(crack_results["0"]["cracks"], crack_results["90"]["cracks"])
print("Done. Crack counts per frame (0d + 90d):")
for name, c0, c90 in zip(
    FRAME_NAMES, crack_results["0"]["metrics"]["crack_count"], crack_results["90"]["metrics"]["crack_count"]
):
    print(f"  {name}: 0deg={int(c0)}, 90deg={int(c90)}")

## 3) Preprocess once for delamination

Both edge and diffuse delamination detection run on *normalized* frames —
see the [Image Pre-processing docs](../docs/source/Image_pre_processing.rst)
for what this does (ratio division against a baseline frame). We do this
once and reuse the cached, normalized frames in every slider callback
below, so only the fast threshold/geometry stage re-runs interactively.

In [ ]:
detector = DelaminationDetector(specimen, interface, save_preprocess_outputs=False)
cache_paths = detector.preprocess_stack_to_disk(
    specimen.image_stack_full,
    key="interactive_notebook",
    reference_mode="static",
)["cache_paths"]
print(f"Preprocessed and cached {len(cache_paths)} frames.")

## Section 1 — Interactive Crack Detection

- **`crack_width_px`** — expected crack width in pixels; drives the
  morphological width used to isolate cracks from background texture.
- **`min_crack_size_px`** — minimum segment length to keep; raise this to
  discard short/spurious detections.

Click **Run Interact** to re-run crack detection with your chosen values
(both 0° and 90° families) and see the overlay on your chosen frame.

In [ ]:
def run_crack_detection(frame_name="0378_sc.png", crack_width_px=8, min_crack_size_px=20, orientation="both"):
    idx = FRAME_INDEX[frame_name]
    print(f"Running crack_eval_crossply(crack_width_px={crack_width_px}, min_crack_size_px={min_crack_size_px})... (~15-25s)")
    results = crack_eval_crossply(
        specimen,
        crack_width_px=float(crack_width_px),
        min_crack_size_px=float(min_crack_size_px),
        export_images=False,
        save_cracks=False,
    )
    frame = specimen.image_stack_full[idx]

    fig, ax = plt.subplots(figsize=(11, 3.4))
    ax.imshow(frame, cmap="gray", vmin=0, vmax=1)
    colors = {"0": "tab:red", "90": "tab:cyan"}
    orientations = ["0", "90"] if orientation == "both" else [orientation]
    counts = {}
    for orient in orientations:
        segs = results[orient]["cracks"][idx]
        counts[orient] = len(segs)
        for seg in segs:
            (r0, c0), (r1, c1) = seg
            ax.plot([c0, c1], [r0, r1], color=colors[orient], linewidth=1.2)
    title = ", ".join(f"{o} deg: {n} cracks" for o, n in counts.items())
    ax.set_title(f"{frame_name} -- {title}")
    ax.axis("off")
    plt.show()


interact_manual(
    run_crack_detection,
    frame_name=widgets.Dropdown(options=FRAME_NAMES, value="0378_sc.png", description="Frame"),
    crack_width_px=widgets.IntSlider(min=2, max=20, step=1, value=8, description="crack_width_px"),
    min_crack_size_px=widgets.IntSlider(min=5, max=100, step=5, value=20, description="min_crack_size_px"),
    orientation=widgets.Dropdown(options=["both", "0", "90"], value="both", description="orientation"),
);

## Section 2 — Interactive Edge Delamination

Runs `detector.edge.detect_primary` on a single selected frame (no
frame-to-frame latching here, so you see each frame's detection in
isolation). Real ground truth: `src/deladect/detection/delamination.py`.

- **`hard_floor`** — normalized brightness gate; pixels brighter than this
  (after smoothing) are excluded from candidates regardless of threshold.
  Lower it to detect more (darker-only) area; raise it to detect less.
- **`window_edge`** — width (px) of the horizontally-elongated smoothing
  window used before thresholding.
- **`seed_ratio`** — fraction of the (half-)frame height used as the
  seed region at the free edge that growth must connect back to.
- **`post_threshold_closing_scale`** — morphological closing radius, as a
  multiple of `avg_crack_width_px`, applied after thresholding.

These update live as you drag (edge detection on one frame takes well
under a second).

In [ ]:
def run_edge_detection(frame_name="0378_sc.png", hard_floor=0.90, window_edge_x=60, seed_ratio=0.01, closing_scale=2.5):
    idx = FRAME_INDEX[frame_name]
    single_cache = [cache_paths[idx]]
    result = detector.edge.detect_primary(
        processed_cache_paths=single_cache,
        params={
            "hard_floor": hard_floor,
            "window_edge": (0, int(window_edge_x)),
            "seed_ratio": seed_ratio,
            "post_threshold_closing_scale": closing_scale,
        },
    )
    mask = next(iter(result["masks"].values()))
    frame = specimen.image_stack_full[idx]

    fig, axes = plt.subplots(1, 2, figsize=(12, 3.4))
    axes[0].imshow(frame, cmap="gray", vmin=0, vmax=1)
    axes[0].set_title(f"{frame_name} (raw)")
    axes[0].axis("off")

    axes[1].imshow(frame, cmap="gray", vmin=0, vmax=1)
    overlay = np.zeros((*mask.shape, 4))
    overlay[mask] = [1, 0, 0, 0.45]
    axes[1].imshow(overlay)
    axes[1].set_title(f"edge mask -- {mask.mean() * 100:.1f}% of frame")
    axes[1].axis("off")
    plt.tight_layout()
    plt.show()


interact(
    run_edge_detection,
    frame_name=widgets.Dropdown(options=FRAME_NAMES, value="0378_sc.png", description="Frame"),
    hard_floor=widgets.FloatSlider(min=0.5, max=0.99, step=0.01, value=0.90, continuous_update=False, description="hard_floor"),
    window_edge_x=widgets.IntSlider(min=10, max=120, step=5, value=60, continuous_update=False, description="window_edge"),
    seed_ratio=widgets.FloatSlider(min=0.005, max=0.05, step=0.005, value=0.01, continuous_update=False, description="seed_ratio"),
    closing_scale=widgets.FloatSlider(min=0.5, max=5.0, step=0.5, value=2.5, continuous_update=False, description="closing_scale"),
);

## Section 3 — Interactive Diffuse Delamination

Runs `detector.diffuse.diffuse_delamination` on a single selected frame,
using that frame's own crack segments (from Section 1's baseline run) to
define local regions of interest.

- **`diffuse_dx` / `diffuse_dy`** — half-width/half-height (px) of the
  rectangular ROI grown around each crack segment.
- **`hard_floor`** — same normalized brightness gate as edge detection,
  applied within each ROI.
- **`post_threshold_closing_scale`** — morphological closing radius after
  thresholding, same convention as edge detection.

This is the fastest section (diffuse detection on one frame is a few tens
of milliseconds), so sliders update live and continuously.

In [ ]:
def run_diffuse_detection(frame_name="0378_sc.png", diffuse_dx=20.0, diffuse_dy=20.0, hard_floor=0.90, closing_scale=2.5):
    idx = FRAME_INDEX[frame_name]
    single_cache = [cache_paths[idx]]
    single_cracks = [cracks[idx]]
    result = detector.diffuse.diffuse_delamination(
        cracks=single_cracks,
        processed_cache_paths=single_cache,
        params={
            "diffuse_dx": diffuse_dx,
            "diffuse_dy": diffuse_dy,
            "hard_floor": hard_floor,
            "post_threshold_closing_scale": closing_scale,
        },
    )
    mask = next(iter(result["masks"].values()))
    frame = specimen.image_stack_full[idx]

    fig, axes = plt.subplots(1, 2, figsize=(12, 3.4))
    axes[0].imshow(frame, cmap="gray", vmin=0, vmax=1)
    axes[0].set_title(f"{frame_name} (raw)")
    axes[0].axis("off")

    axes[1].imshow(frame, cmap="gray", vmin=0, vmax=1)
    overlay = np.zeros((*mask.shape, 4))
    overlay[mask] = [0, 1, 0, 0.45]
    axes[1].imshow(overlay)
    axes[1].set_title(f"diffuse mask -- {mask.mean() * 100:.1f}% of frame")
    axes[1].axis("off")
    plt.tight_layout()
    plt.show()


interact(
    run_diffuse_detection,
    frame_name=widgets.Dropdown(options=FRAME_NAMES, value="0378_sc.png", description="Frame"),
    diffuse_dx=widgets.FloatSlider(min=5, max=60, step=5, value=20, description="diffuse_dx"),
    diffuse_dy=widgets.FloatSlider(min=5, max=60, step=5, value=20, description="diffuse_dy"),
    hard_floor=widgets.FloatSlider(min=0.5, max=0.99, step=0.01, value=0.90, description="hard_floor"),
    closing_scale=widgets.FloatSlider(min=0.5, max=5.0, step=0.5, value=2.5, description="closing_scale"),
);

## What to notice

- On sample-1, `hard_floor` and `diffuse_dx`/`diffuse_dy` (ROI size) have
  the strongest effect on how much area gets flagged — try dragging them
  to the extremes on `0378_sc.png` and watch the mask grow or nearly
  disappear.
- `post_threshold_closing_scale` and `seed_ratio` matter more for cleaning
  up *small or marginal* detections (try them on `0189_sc.png`, which has
  much less damage than `0378_sc.png`) than for an already-large,
  well-connected mask.
- Frames `0002_sc.png`/`0092_sc.png` show essentially no delamination with
  any of these settings — that's expected, not a bug: those frames are
  early in the test, before damage has developed.

## Next steps

- [`docs/source/delamination.rst`](../docs/source/delamination.rst) for
  the full parameter tables and algorithm descriptions.
- [`docs/source/Image_pre_processing.rst`](../docs/source/Image_pre_processing.rst)
  for what `preprocess_stack_to_disk` does before any of this runs.
- [`docs/source/examples/getting_started.rst`](../docs/source/examples/getting_started.rst)
  for the full non-interactive workflow (including saving results).